In [18]:
from src.data.load_data import load_reviews

df_reviews_rag = load_reviews()

print(f"Total de registros: {len(df_reviews_rag)}")
print(
    "Pedidos únicos:",
    df_reviews_rag["order_id"].nunique()
)

display(df_reviews_rag.head())

Total de registros: 40950
Pedidos únicos: 40809


,review_id,order_id,review_score,review_comment_message
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,aparelho eficiente. no site a marca do aparelh...
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,"Vendedor confiável, produto ok e entrega antes..."


In [19]:
from src.rag.documents import create_documents

documents = create_documents(df_reviews_rag)

print("Quantidade de documentos:", len(documents))
print("\nExemplo de conteúdo:")
print(documents[0].page_content)

print("\nMetadados:")
print(documents[0].metadata)

Quantidade de documentos: 40950

Exemplo de conteúdo:
passage: Recebi bem antes do prazo estipulado.

Metadados:
{'review_id': 'e64fb393e7b32834bb789ff8bb30750e', 'order_id': '658677c97b385a9be170737859d3511b', 'review_score': 5}


In [20]:
from src.rag.embedding import load_embedding_model

embeddings = load_embedding_model()

textos = [
    "passage: Meu pedido chegou muito atrasado.",
    "passage: A entrega demorou mais do que o esperado."
]

vetores = embeddings.embed_documents(textos)

print("Quantidade de vetores:", len(vetores))
print("Dimensões do primeiro vetor:", len(vetores[0]))
print("Primeiros valores:", vetores[0][:5])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 364.39it/s]


Quantidade de vetores: 2
Dimensões do primeiro vetor: 768
Primeiros valores: [0.0059566805139184, 0.03336201608181, -0.024319935590028763, 0.002808919409289956, 0.010610085912048817]


In [21]:
pergunta = "query: Quais clientes reclamaram de atrasos na entrega?"

vetor_pergunta = embeddings.embed_query(pergunta)

print("Dimensões do vetor da pergunta:", len(vetor_pergunta))

Dimensões do vetor da pergunta: 768


In [23]:
from pathlib import Path
from src.rag.embedding import load_embedding_model
from src.rag.documents import create_documents
from src.rag.vector_store import create_vector_store

PROJECT_ROOT = Path.cwd().parent

documents = create_documents(df_reviews_rag)
embeddings = load_embedding_model()

vector_store = create_vector_store(
    documents=documents,
    embeddings=embeddings,
    persist_directory=(
        PROJECT_ROOT / "data" / "processed" / "chroma_rebuilt"
    ),
)

print("Indexação concluída!")
print("Documentos processados:", len(documents))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 384.98it/s]


Indexação concluída!
Documentos processados: 40950


In [24]:
pergunta = "query: Quais clientes reclamaram sobre a qualidade do item?"

resultados = vector_store.similarity_search(
    pergunta,
    k=5,
)

for i, documento in enumerate(resultados, start=1):
    print(f"\nResultado {i}")
    print("Comentário:", documento.page_content)
    print("Nota:", documento.metadata["review_score"])
    print("Pedido:", documento.metadata["order_id"])


Resultado 1
Comentário: passage: O produto veio em duas oportunidades, mal embalado, com diversas avarias na pintura, algumas peças tortas. Precisam melhorar a embalagem e colocar avisos de "Frágil" ou não pisar.
Nota: 1
Pedido: ea884b5a25f86e605e1750bd936e0122

Resultado 2
Comentário: passage: O produto chegou mal embalado, caixa rasgada e molhada, possui defeitos na costura e tem um péssimo ajuste no banco do veículo (Voyage G6). Não recomendo!!
Nota: 1
Pedido: d23555007fbb6b414f8bb5ae6854782a

Resultado 3
Comentário: passage: O produto veio repleto de avarias e as peças não se encaixam devidamente, o que foi constatado por montador profissional. A qualidade também não é a indicada. Lastimável!
Nota: 1
Pedido: b78d95a93c70ace32ceafee76789a2f1

Resultado 4
Comentário: passage: Não recebi o produto e ainda não tive nenhuma resposta da minha reclamação. E o pior é que seria presente de 1 ano do avô da criança, o qual pagou mais de R$500,00 pela mesma.
Nota: 1
Pedido: 6e6bacc052a79d14a6

### Intergrando funções

In [25]:
from src.rag.chain import load_llm, ask_rag

llm = load_llm()

In [26]:
resultado = ask_rag(
    question="Quais problemas de entrega são relatados pelos clientes?",
    vector_store=vector_store,
    llm=llm,
    k=5,
)

print(resultado["answer"])

Os problemas de entrega relatados pelos clientes são:

- Produto não entregue: [Avaliação 2], [Avaliação 4]
- Problemas com a confirmação do pedido, resultando em atrasos na entrega: [Avaliação 3]
- Produto danificado durante a entrega (embalagem inadequada e vidros vazando): [Avaliação 5]


In [27]:
for fonte in resultado["sources"]:
    print("-" * 60)
    print("Review ID:", fonte["review_id"])
    print("Pedido:", fonte["order_id"])
    print("Nota:", fonte["review_score"])
    print("Comentário:", fonte["comment"])

------------------------------------------------------------
Review ID: 98938945a4aa26f21a5d2c3a6112b745
Pedido: 577b75217094b43a3e032b217d67f70f
Nota: 5
Comentário: Não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz uma reclamação e não tive resposta. Ainda estou aguardando.
------------------------------------------------------------
Review ID: 7c8154d29d239b02ad0e8f2dcdf3261a
Pedido: cacb1996e2de4ef6334c5223259544e0
Nota: 1
Comentário: Péssimo o procedimento. Vergonha. Nada entregue até agora. Absurdo e descaso com cliente.
------------------------------------------------------------
Review ID: bf8dad1ed02a48807cbe4027625c5bc8
Pedido: 43bd50a07cc1758b29b32804f1a6d552
Nota: 1
Comentário: Já recebi diferentes e-mails: confirmação da entrega, dificuldades para confirmar pedido, pedido ainda não confirmado e mais uma vez, de confirmação da entrega. Será que chega?
------------------------------------------------------

### Tempo de processamento da pergunta

In [29]:

from time import perf_counter

pergunta = "Quais problemas de entrega são relatados pelos clientes?"

# Etapa 1: recuperação
inicio = perf_counter()

documentos = vector_store.similarity_search(
    f"query: {pergunta}",
    k=5,
)

tempo_busca = perf_counter() - inicio
print(f"Tempo de recuperação: {tempo_busca:.2f} segundos")
print(f"Documentos recuperados: {len(documentos)}")

# Etapa 2: geração
contexto = "\n\n".join(
    f"[Avaliação {i}]\n"
    f"Comentário: {doc.page_content.removeprefix('passage: ')}\n"
    f"Nota: {doc.metadata.get('review_score', 'N/A')}"
    for i, doc in enumerate(documentos, start=1)
)

prompt = f"""
Responda em português usando apenas as avaliações fornecidas.
Se as evidências forem insuficientes, informe isso.
Seja objetivo e cite as avaliações relevantes.

Pergunta: {pergunta}

Avaliações:
{contexto}
"""

inicio = perf_counter()

resposta = llm.invoke(prompt)

tempo_llm = perf_counter() - inicio
print(f"Tempo de geração: {tempo_llm:.2f} segundos")
print("\nResposta:")
print(resposta.content)


Tempo de recuperação: 0.95 segundos
Documentos recuperados: 5
Tempo de geração: 573.43 segundos

Resposta:
Os problemas de entrega relatados pelos clientes são:

- Produto não entregue (Avaliação 2, 3, 4 e 5)
- Produto mal embalado com vidros vazando (Avaliação 5)
